# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is defined by a Croissant schema hosted online:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Let's load the FAIR^2 dataset Croissant schema and print general project metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}\n")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}\n")
print("Description:")
print(metadata.description)

print("\nKeywords:")
try:
    for k in metadata.keywords:
        print(f"- {k}")
except Exception:
    print("None")


## 2. Data Overview

In Croissant, **record sets** group related records (tabular data). Each record set has fields (columns), each with a unique `@id`.

Let's list available record sets, their `@id`s and the fields available in each.

In [ ]:
# List all record sets and their fields, referenced by @id

all_record_sets = list(dataset.record_sets)
if not all_record_sets:
    print('No record sets found in the schema.')
else:
    print(f"Found {len(all_record_sets)} record set(s):\n")
    for rs in all_record_sets:
        print(f"Record set name: {rs.name}")
        print(f"@id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - {field.name} (@id={field.id})")
        else:
            print('  No fields found.')
        print()


## 3. Data Extraction

Let's load records from all available record sets into pandas DataFrames, referencing all entities by their `@id`. If multiple record sets exist, we will load and display a preview for each.

In [ ]:
# For all discovered record sets, load data by @id

dataframes = {}
loaded_rs_ids = []

for rs in all_record_sets:
    print(f"Loading records for record set: {rs.name} (@id={rs.id})")
    try:
        records = list(dataset.records(record_set=rs.id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs.id] = df
            loaded_rs_ids.append(rs.id)
            print(f"Loaded {len(df)} records. Columns by @id:")
            print(list(df.columns))
            display(df.head())
        else:
            print("No records returned from this record set.")
    except Exception as e:
        print(f"Failed to load: {e}")
    print()


## 4. Exploratory Data Analysis (EDA)

Perform data processing: 
- Filter records on a numeric field
- Normalize that field
- Group by a categorical field (if available)

If multiple record sets, choose the first for demonstration.

In [ ]:
# Pick the first loaded record set and try to find a numeric and a group/categorical field

import numpy as np

if loaded_rs_ids:
    record_set_id = loaded_rs_ids[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}\nColumns: {list(df.columns)}\n")

    # Try to infer a numeric field and a group field
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        # Check dtype by sampling
        sample_val = df[col].dropna().head(1)
        if not sample_val.empty:
            v = sample_val.values[0]
            if isinstance(v, (int, float, np.integer, np.floating)):
                numeric_field_id = col
                break
            try:
                # Some values may be strings containing numerics
                float(v)
                numeric_field_id = col
                break
            except Exception:
                pass

    if numeric_field_id:
        # Cast column to float if possible
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        # Find a group/categorical field (not the numeric)
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < len(df) // 2:
                group_field_id = col
                break

        threshold = np.nanmean(df[numeric_field_id]) if np.nanmean(df[numeric_field_id]) is not np.nan else 0
        print(f"Filtering rows where {numeric_field_id} > {threshold:.3f}")
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {len(filtered_df)} records.")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized column '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by categorical (if found)
        if group_field_id:
            print(f"\nGrouped by '{group_field_id}':")
            group_stats = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(group_stats.head())
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No record sets with loaded data for EDA.")


## 5. Visualization

Visualize data distributions based on the numeric field in the selected record set, grouped if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if loaded_rs_ids and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set '{record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")


## 6. Conclusion

We loaded the FAIR^2 dataset using `mlcroissant`, reviewed available record sets and their schema (referenced by their unique `@id`), and demonstrated a typical data analysis workflow, including normalization and visualization. 

All data access was by entity `@id`, as recommended for robust, reproducible data science workflows.

To go further:
- Deepen statistical analysis on other record sets
- Explore relationships between fields using their `@id`
- Integrate data from multiple record sets for holistic analysis

For more, see [mlcroissant documentation](https://github.com/mlcommons/croissant) and the [FAIR^2 dataset page](https://doi.org/10.71728/senscience.y7m0-f273).